# LAB3 Regression & Classification 

---
นำเข้า Libraries

In [ ]:
# ติดตั้งไลบรารีสำหรับการวิเคราะห์ข้อมูล การสร้างโมเดล และการพล็อตกราฟ
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# นำเข้าไลบรารีที่จำเป็นทั้งหมด
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# ตั้งค่าฟอนต์และการแสดงผลกราฟ
import matplotlib.font_manager as fm

# ระบุ Path ไปยังไฟล์ฟอนต์ภาษาไทยในเครื่อง Windows โดยตรง
font_path = "C:/Windows/Fonts/leelawui.ttf"  # ฟอนต์ Leelawadee UI
if not Path(font_path).exists():
    font_path = "C:/Windows/Fonts/tahoma.ttf"  # กรณีไม่มี ให้ใช้ Tahoma แทน

font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False  # ป้องกันเครื่องหมายลบเพี้ยน

# ล้าง Font Cache ของ Matplotlib
fm.fontManager.addfont(font_path)

---

การเตรียมข้อมูลและการลดมิติด้วย PCA

In [ ]:
# 1. โหลดข้อมูลภาพใบหน้า (UTKFace จาก age_gender.csv หรือจำลองข้อมูล Synthetic Data)
data_path = Path("age_gender.csv")

if data_path.exists():
    print("กำลังโหลดข้อมูลจากไฟล์ age_gender.csv...")
    df = pd.read_csv(data_path).dropna()
    
    # กรองเฉพาะแถวที่พิกเซลมีความยาวครบ 2,304 ค่า (48x48) ป้องกัน Error
    df['pixels_list'] = df['pixels'].apply(lambda x: np.fromstring(x, dtype=float, sep=' '))
    df = df[df['pixels_list'].apply(lambda x: len(x) == 2304)]
    
    # สุ่ม 2,000 ตัวอย่างเพื่อความรวดเร็วในการประมวลผล
    df = df.sample(n=min(2000, len(df)), random_state=42)
    
    X_raw = np.vstack(df['pixels_list'].values)
    y_age = df['age'].values.astype(float)
    y_gender = df['gender'].values.astype(int)  # 0 = ชาย, 1 = หญิง
else:
    print("ไม่พบไฟล์ age_gender.csv ทำการจำลองชุดข้อมูลพิกเซลใบหน้า (Synthetic Data)...")
    np.random.seed(42)
    n_samples, n_features = 1000, 2304  # เสมือนรูปภาพใบหน้าขนาด 48x48 พิกเซล
    X_raw = np.random.randn(n_samples, n_features)
    y_age = np.clip(25 + 12 * X_raw[:, 0] - 6 * X_raw[:, 1] + 8 * X_raw[:, 2] + np.random.normal(0, 3, n_samples), 1, 90)
    y_gender = (1.5 * X_raw[:, 0] - 2.0 * X_raw[:, 1] + np.random.normal(0, 1, n_samples) > 0).astype(int)

# 2. Standardization ปรับสเกลข้อมูลพิกเซลให้ Mean=0, Std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. PCA ลดมิติข้อมูลเหลือ 50 Principal Components
n_components = 50
pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("ขนาดข้อมูลพิกเซลดั้งเดิม:", X_raw.shape)
print("ขนาดข้อมูลหลังสกัดด้วย PCA:", X_pca.shape)
print(f"ความแปรปรวนรวมที่อธิบายได้ (Total Explained Variance): {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

---

LAB 3.1: Regression (การทำนายอายุ)

In [ ]:
# 3.1.1 Simple Linear Regression: ถดถอยเชิงเส้นอย่างง่ายโดยใช้ 1 Feature (PC1)
# แบ่งข้อมูลสำหรับงาน Regression (Train 80% / Test 20%)
X_train_reg, X_test_reg, y_train_age, y_test_age = train_test_split(
    X_pca, y_age, test_size=0.2, random_state=42
)

slr = LinearRegression()
slr.fit(X_train_reg[:, :1], y_train_age)

# ทำนายผลบนชุด Train และ Test
y_pred_slr_train = slr.predict(X_train_reg[:, :1])
y_pred_slr_test = slr.predict(X_test_reg[:, :1])

# วาดกราฟเส้นตรงถดถอย Simple Linear Regression
plt.figure(figsize=(8, 4.5))
plt.scatter(X_test_reg[:, 0], y_test_age, alpha=0.5, color='royalblue', label='อายุจริง (Actual Age)')
plt.plot(X_test_reg[:, 0], y_pred_slr_test, color='crimson', linewidth=2, label='เส้นถดถอย (SLR Line)')
plt.title('3.1.1 Simple Linear Regression: PC1 vs Age')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('อายุ (Age)')
plt.legend()
plt.show()

In [ ]:
# 3.1.2 Multiple Linear Regression: ถดถอยเชิงเส้นพหุคูณโดยใช้หลาย Features (PC1 - PC50) 
mlr = LinearRegression()
mlr.fit(X_train_reg, y_train_age)

# ทำนายผลบนชุด Train และ Test
y_pred_mlr_train = mlr.predict(X_train_reg)
y_pred_mlr_test = mlr.predict(X_test_reg)

# วาดกราฟเปรียบเทียบ Actual vs Predicted Age
plt.figure(figsize=(8, 4.5))
plt.scatter(y_test_age, y_pred_mlr_test, alpha=0.6, color='forestgreen')
plt.plot([y_test_age.min(), y_test_age.max()], [y_test_age.min(), y_test_age.max()], 'r--', lw=2, label='เส้นพอดีสมบูรณ์แบบ (Ideal Fit)')
plt.title('3.1.2 Multiple Linear Regression: Actual vs Predicted Age')
plt.xlabel('อายุจริง (Actual Age)')
plt.ylabel('อายุที่ทำนายได้ (Predicted Age)')
plt.legend()
plt.show()

---

LAB 3.2: Classification (การจำแนกเพศ)

In [ ]:
# 3.2.1 Preparing Classification Data (แบ่งข้อมูลแบบ Stratified) 
X_train_clf, X_test_clf, y_train_gen, y_test_gen = train_test_split(
    X_pca, y_gender, test_size=0.2, random_state=42, stratify=y_gender
)

# 3.2.2 Logistic Regression & Gender Prediction 
clf = LogisticRegression(random_state=42, max_iter=500)
clf.fit(X_train_clf, y_train_gen)

# ทำนายคลาสและความน่าจะเป็น
y_pred_gen_train = clf.predict(X_train_clf)
y_pred_gen_test = clf.predict(X_test_clf)
y_prob_gen_test = clf.predict_proba(X_test_clf)[:, 1]

print("--- 3.2.2 Classification Report (การจำแนกเพศ) ---")
print(classification_report(y_test_gen, y_pred_gen_test, target_names=['ชาย (0)', 'หญิง (1)']))

In [ ]:
# 3.2.3 Confusion Matrix & ROC Curve 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test_gen, y_pred_gen_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['ทำนาย: ชาย', 'ทำนาย: หญิง'],
            yticklabels=['จริง: ชาย', 'จริง: หญิง'])
axes[0].set_title('Confusion Matrix')

# 2. กราฟ ROC Curve และ AUC
fpr, tpr, _ = roc_curve(y_test_gen, y_prob_gen_test)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc_val:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--', lw=1.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2.4 Decision Boundary Visualization (บนระนาบ 2 มิติ: PC1 vs PC2)
clf_2d = LogisticRegression(random_state=42)
clf_2d.fit(X_train_clf[:, :2], y_train_gen)

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))

Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 5.5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
scatter = plt.scatter(X_test_clf[:, 0], X_test_clf[:, 1], c=y_test_gen, cmap='coolwarm', edgecolors='k', alpha=0.8)
plt.title('3.2.4 Logistic Regression Decision Boundary (PC1 vs PC2)')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.legend(*scatter.legend_elements(), title='เพศ (0=ชาย, 1=หญิง)')
plt.grid(True, alpha=0.3)
plt.show()

---

LAB 3.3: Model Comparison (การเขียนโค้ดเปรียบเทียบประสิทธิภาพโมเดล)

In [ ]:
# 3.3.1 Simple vs Multiple Linear Regression Comparison 
# โค้ดเปรียบเทียบผลลัพธ์ระหว่างโมเดล 1 Feature (Simple) และ 50 Features (Multiple)
slr_train_mse, slr_test_mse = mean_squared_error(y_train_age, y_pred_slr_train), mean_squared_error(y_test_age, y_pred_slr_test)
mlr_train_mse, mlr_test_mse = mean_squared_error(y_train_age, y_pred_mlr_train), mean_squared_error(y_test_age, y_pred_mlr_test)

df_reg_comp = pd.DataFrame({
    'Model': ['Simple Linear Regression (PC1)', 'Multiple Linear Regression (50 PCs)'],
    'Train MSE': [slr_train_mse, mlr_train_mse],
    'Test MSE': [slr_test_mse, mlr_test_mse],
    'Test MAE': [mean_absolute_error(y_test_age, y_pred_slr_test), mean_absolute_error(y_test_age, y_pred_mlr_test)],
    'Train R²': [r2_score(y_train_age, y_pred_slr_train), r2_score(y_train_age, y_pred_mlr_train)],
    'Test R²': [r2_score(y_test_age, y_pred_slr_test), r2_score(y_test_age, y_pred_mlr_test)]
})

print("=== 3.3.1 ผลการเปรียบเทียบ Simple vs Multiple Linear Regression ===")
print(df_reg_comp.to_string(index=False))

# วาดกราฟแท่งเปรียบเทียบ R² Score และ Test MSE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x='Model', y='Test R²', data=df_reg_comp, ax=axes[0], palette='Blues_d')
axes[0].set_title('เปรียบเทียบค่าความแม่นยำ R² (ยิ่งสูงยิ่งดี)')

sns.barplot(x='Model', y='Test MSE', data=df_reg_comp, ax=axes[1], palette='Reds_d')
axes[1].set_title('เปรียบเทียบความคลาดเคลื่อน MSE (ยิ่งต่ำยิ่งดี)')

plt.tight_layout()
plt.show()

In [ ]:
# 3.3.2 Training vs Testing Performance (การตรวจเช็ก Overfitting)
# โค้ดเปรียบเทียบคะแนน Train กับ Test ของทั้ง 2 โมเดลเพื่อดู Generalization Gap
perf_check = pd.DataFrame({
    'Model Task': ['Regression (Multiple Linear - MSE)', 'Classification (Logistic - Accuracy)'],
    'Train Score': [
        mean_squared_error(y_train_age, y_pred_mlr_train),
        accuracy_score(y_train_gen, y_pred_gen_train)
    ],
    'Test Score': [
        mean_squared_error(y_test_age, y_pred_mlr_test),
        accuracy_score(y_test_gen, y_pred_gen_test)
    ]
})
perf_check['Difference (Train vs Test)'] = np.abs(perf_check['Train Score'] - perf_check['Test Score'])

print("=== 3.3.2 ตารางเปรียบเทียบผล Training vs Testing ===")
print(perf_check.to_string(index=False))

In [ ]:
# 3.3.3 Regression vs Classification Code Comparison
# 1. สร้างตารางตัวอย่างเปรียบเทียบ Output จริงกับ Output ที่ทำนายได้ 5 แถวแรกของทั้ง 2 งาน
df_task_compare = pd.DataFrame({
    'Sample ID': [f'Sample #{i+1}' for i in range(5)],
    'Reg_True_Age': y_test_age[:5].round(1),
    'Reg_Pred_Age (Continuous)': y_pred_mlr_test[:5].round(2),
    'Clf_True_Gender': y_test_gen[:5],
    'Clf_Pred_Gender (Class 0/1)': y_pred_gen_test[:5],
    'Clf_Prob_Female (P(y=1))': y_prob_gen_test[:5].round(4)
})

print("=== 3.3.3 ตัวอย่างการเปรียบเทียบชนิดของ Output ระหว่าง Regression และ Classification ===")
print(df_task_compare.to_string(index=False))

# 2. พล็อตกราฟเปรียบเทียบพฤติกรรมการทำนายระหว่าง Regression (ความคลาดเคลื่อน Residuals) vs Classification (ความน่าจะเป็น Sigmoid)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ฝั่ง Regression: การกระจายตัวของค่าความคลาดเคลื่อน (Residuals Distribution)
residuals = y_test_age - y_pred_mlr_test
sns.histplot(residuals, kde=True, color='teal', ax=axes[0])
axes[0].set_title('Regression: การกระจายตัวของ Residuals (Actual - Predicted)')
axes[0].set_xlabel('ค่าความคลาดเคลื่อน (Years)')
axes[0].set_ylabel('จำนวนตัวอย่าง (Count)')

# ฝั่ง Classification: การกระจายตัวของค่าความน่าจะเป็น (Predicted Probability Distribution)
sns.histplot(y_prob_gen_test, kde=True, color='darkorange', ax=axes[1], bins=15)
axes[1].axvline(0.5, color='red', linestyle='--', label='เกณฑ์ตัดคลาส (Threshold = 0.5)')
axes[1].set_title('Classification: การกระจายตัวของความน่าจะเป็น P(Gender=Female)')
axes[1].set_xlabel('ค่าความน่าจะเป็น (Probability 0.0 - 1.0)')
axes[1].set_ylabel('จำนวนตัวอย่าง (Count)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 3.3.4 Model Performance Metrics Summary 
# รวบรวมตัวชี้วัดประสิทธิภาพของทั้งสองงานลงในตารางสรุปเดียว
summary_metrics = pd.DataFrame({
    'Machine Learning Task': [
        'Regression (Age Prediction)',
        'Regression (Age Prediction)',
        'Regression (Age Prediction)',
        'Classification (Gender Prediction)',
        'Classification (Gender Prediction)',
        'Classification (Gender Prediction)',
        'Classification (Gender Prediction)'
    ],
    'Model': [
        'Simple Linear Regression',
        'Multiple Linear Regression',
        'Multiple Linear Regression',
        'Logistic Regression',
        'Logistic Regression',
        'Logistic Regression',
        'Logistic Regression'
    ],
    'Metric Type': [
        'Test MSE',
        'Test MSE',
        'Test R² Score',
        'Test Accuracy',
        'Test Precision',
        'Test Recall (F1-Score)',
        'ROC-AUC'
    ],
    'Value': [
        f"{mean_squared_error(y_test_age, y_pred_slr_test):.4f}",
        f"{mean_squared_error(y_test_age, y_pred_mlr_test):.4f}",
        f"{r2_score(y_test_age, y_pred_mlr_test):.4f}",
        f"{accuracy_score(y_test_gen, y_pred_gen_test):.4f}",
        f"{precision_score(y_test_gen, y_pred_gen_test):.4f}",
        f"{f1_score(y_test_gen, y_pred_gen_test):.4f}",
        f"{roc_auc_val:.4f}"
    ]
})

print("=== 3.3.4 ตารางสรุปตัวชี้วัดประสิทธิภาพโมเดลทั้งหมด (Metrics Summary) ===")
print(summary_metrics.to_string(index=False))